# 🧹 GIAI ĐOẠN 2: TIỀN XỬ LÝ & CHUẨN HÓA VĂN BẢN TIẾNG VIỆT (TV1)
**Thực hiện:** TV1 - Hoàng Hôn (Trưởng nhóm)

### Mục tiêu:
1. Ghép các trường văn bản (`Title`, `What I liked`, `Suggestions for improvement`).
2. Làm sạch cơ bản: chuẩn hóa Unicode NFC, xử lý emoji/emojicon, dịch teencode, sửa từ sai chính tả.
3. Trích xuất đặc trưng Lexicon cảm xúc (`pos_w`, `neg_w`, `sentiment_ratio`).
4. Tách từ tiếng Việt (`underthesea`) và loại bỏ stopwords.
5. Gán nhãn cảm xúc 3 lớp (`Positive`, `Neutral`, `Negative`) và xuất tập dữ liệu sạch.


## 1. Import các thư viện và Module Preprocessing


In [1]:
import sys, os
sys.path.append("..")
import pandas as pd
import numpy as np
from tqdm import tqdm
from src.preprocessing import TextPreprocessor

tqdm.pandas()
print("✅ Đã nạp thành công các thư viện!")


✅ Đã nạp thành công các thư viện!


## 2. Đọc dữ liệu thô từ ITviec Reviews


In [2]:
raw_path = "../data/raw/Reviews.xlsx"
df = pd.read_excel(raw_path)
print(f"Tổng số mẫu đánh giá: {len(df):,} | Số cột: {df.shape[1]}")
display(df.head(2))


Tổng số mẫu đánh giá: 8,417 | Số cột: 13


,id,Company Name,Cmt_day,Title,What I liked,Suggestions for improvement,Rating,Salary & benefits,Training & learning,Management cares about me,Culture & fun,Office & workspace,Recommend?
0,4,Accenture,March 2025,"Môi trường thoải mái, ít áp lực, có thể làm vi...","Môi trường thoải mái, ít áp lực, có thể làm vi...",Cần đánh giá KPI khách quan và phân chia công ...,4,4,4,4,4,4,Yes
1,4,Accenture,January 2025,Công ty trẻ trung năng động,"Cơ sở vật chất đẹp, công ty trẻ trung năng độn...",Nên nghiên cứu lại range lương của thì trường....,4,3,4,4,4,5,Yes


## 3. Ghép các trường văn bản


In [3]:
title = df["Title"].fillna("").astype(str)
liked = df["What I liked"].fillna("").astype(str)
suggest = df["Suggestions for improvement"].fillna("").astype(str)

df["raw_review_text"] = title + " . " + liked + " . " + suggest
print("Ví dụ văn bản thô sau khi gộp:")
print(df["raw_review_text"].iloc[0][:200] + "...")


Ví dụ văn bản thô sau khi gộp:
Môi trường thoải mái, ít áp lực, có thể làm việc hybrid, lương ở mức khá. . Môi trường thoải mái, ít áp lực, có thể làm việc hybrid, lương nên deal tốt từ lúc đầu, vì mức tăng mỗi năm ko cao. Các bene...


## 4. Khởi tạo bộ tiền xử lý và Làm sạch cơ bản (Clean Basic Text)


In [4]:
tp = TextPreprocessor(dict_dir="../data/dictionaries")

# Áp dụng làm sạch cơ bản
df["clean_basic_text"] = df["raw_review_text"].progress_apply(tp.clean_basic_text)
print("Mẫu sau khi làm sạch cơ bản:")
print(df["clean_basic_text"].iloc[0][:200] + "...")


100%|██████████| 8417/8417 [00:01<00:00, 5037.86it/s]

Mẫu sau khi làm sạch cơ bản:
môi trường thoải mái ít áp lực có thể làm việc hybrid lương ở mức khá môi trường thoải mái ít áp lực có thể làm việc hybrid lương nên thỏa thuận tốt từ lúc đầu vì mức tăng mỗi năm không cao các phúc l...


## 5. Trích xuất đặc trưng Thống kê Lexicon Cảm xúc


In [5]:
lex_feats = df.apply(lambda row: tp.calc_sentiment_features(row["clean_basic_text"], raw_text=row["raw_review_text"]), axis=1)
lex_df = pd.DataFrame(list(lex_feats))

for col in lex_df.columns:
    df[col] = lex_df[col]

display(df[["clean_basic_text", "pos_w", "neg_w", "pos_e", "neg_e", "sentiment_ratio"]].head())


,clean_basic_text,pos_w,neg_w,pos_e,neg_e,sentiment_ratio
0,môi trường thoải mái ít áp lực có thể làm việc...,6,3,0,0,0.3333
1,công ty trẻ trung năng động cơ sở vật chất đẹp...,5,1,0,0,0.6667
2,môi trường làm việc thoải mái vui vẻ mội người...,6,1,0,0,0.7143
3,tốt environment for nhân viên mới tốt env for ...,4,0,0,0,1.0000
4,môi trường làm việc thoải mái vui vẻ văn phòng...,12,1,0,0,0.8462


## 6. Làm sạch nâng cao: Tách từ Tiếng Việt (underthesea) & Lọc Stopwords


In [6]:
df["clean_advance_text"] = df["raw_review_text"].progress_apply(
    lambda x: tp.clean_advance_text(x, remove_stopwords=True)
)

print("Mẫu sau khi tách từ và lọc stopwords:")
print(df["clean_advance_text"].iloc[0][:200] + "...")


100%|██████████| 8417/8417 [00:40<00:00, 205.74it/s]

Mẫu sau khi tách từ và lọc stopwords:
môi_trường thoải_mái ít áp_lực có_thể làm_việc hybrid lương môi_trường thoải_mái ít áp_lực có_thể làm_việc hybrid lương thỏa_thuận tốt đầu không cao phúc_lợi cơ_bản đầy_đủ công_ty dịch_vụ nhiều trau_d...


## 7. Gán nhãn Cảm xúc & Thống kê phân bố nhãn


In [7]:
df["sentiment"] = df["Rating"].apply(tp.map_sentiment_label)

sentiment_dist = df["sentiment"].value_counts()
sentiment_pct = df["sentiment"].value_counts(normalize=True) * 100
summary_df = pd.DataFrame({"Số lượng": sentiment_dist, "Tỷ lệ (%)": sentiment_pct.round(2)})

print("📊 Bảng phân bố Sentiment:")
display(summary_df)


📊 Bảng phân bố Sentiment:


,Số lượng,Tỷ lệ (%)
sentiment,,
Positive,6208,73.76
Neutral,1639,19.47
Negative,570,6.77


## 8. Lưu tập dữ liệu đã làm sạch vào `data/processed/`


In [8]:
os.makedirs("../data/processed", exist_ok=True)
out_xlsx = "../data/processed/reviews_cleaned.xlsx"
out_csv = "../data/processed/reviews_cleaned.csv"

df.to_excel(out_xlsx, index=False)
df.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"""🎉 Đã lưu thành công tập dữ liệu sạch tại:
1. {out_xlsx}
2. {out_csv}""")


🎉 Đã lưu thành công tập dữ liệu sạch tại:
1. ../data/processed/reviews_cleaned.xlsx
2. ../data/processed/reviews_cleaned.csv
